# Quantum Approximate Optimization for Soft Robotic Trajectory Planning
---
**Run Cell 1, then Cell 2. No file upload needed.**

### Fixed: GPU out-of-memory crash
The previous version tried N_q=32 on GPU, which needs a 64 GB statevector —
more than a Kaggle T4's 16 GB VRAM. This version caps the GPU problem at
**N_q=20** (T=5, n=4), which only needs 16 MB and runs safely on any GPU.

| Mode | Config | Memory | Time |
|------|--------|--------|------|
| CPU (default) | T=4, n=3, N_q=12 | <1 MB | ~5 min |
| GPU T4 | T=5, n=4, N_q=20 | 16 MB | ~10-20 min |

> To enable GPU: Kaggle sidebar → Settings → Accelerator → **GPU T4 x2**


In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Installs PennyLane AND writes the script to disk.   ║
# ║  Run this FIRST. Safe to re-run.                              ║
# ╚═══════════════════════════════════════════════════════════════╝
import subprocess, sys, os, base64

def _pip(pkg):
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', pkg,
         '--quiet', '--no-warn-script-location'])

try:
    import pennylane as qml
    print(f"PennyLane {qml.__version__} already installed.")
except ImportError:
    print("Installing PennyLane (~30 s) ...")
    _pip('pennylane')
    import pennylane as qml
    print(f"PennyLane {qml.__version__} installed.")

try:
    _pip('pennylane-lightning-gpu')
    import pennylane_lightning
    print("Lightning GPU backend installed.")
except Exception:
    print("Lightning not available — CPU mode is fine (N_q=12).")

SCRIPT_PATH = '/kaggle/working/qaoa_soft_robot.py' if os.path.exists('/kaggle/working') \
              else 'qaoa_soft_robot.py'

SCRIPT_B64 = "..."

script_bytes = base64.b64decode(SCRIPT_B64)
with open(SCRIPT_PATH, 'wb') as f:
    f.write(script_bytes)

print(f"\nScript written to: {SCRIPT_PATH}")
print(f"Size: {os.path.getsize(SCRIPT_PATH)} bytes")
print("\n✅ Cell 1 done. Run Cell 2 to execute the experiment.")


## Run the Experiment


In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Runs the script written by Cell 1.                  ║
# ╚═══════════════════════════════════════════════════════════════╝
import os

SCRIPT_PATH = '/kaggle/working/qaoa_soft_robot.py' if os.path.exists('/kaggle/working') \
              else 'qaoa_soft_robot.py'

assert os.path.exists(SCRIPT_PATH), \
    f"Script not found at {SCRIPT_PATH} — run Cell 1 first!"
print(f"Running {SCRIPT_PATH} ...\n")

%run $SCRIPT_PATH


## View Saved Figures


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, glob

FIGDIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
fig_files = sorted(glob.glob(f'{FIGDIR}/fig*.png'))

print(f"Found {len(fig_files)} figures in {FIGDIR}/")

for fp in fig_files:
    img = mpimg.imread(fp)
    fig, ax = plt.subplots(figsize=(14, 14 * img.shape[0] / img.shape[1]))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(fp), fontsize=10)
    plt.tight_layout()
    plt.show()
